# 🧠 Custom CNN — CIFAR-10 Image Classification

**Dataset:** CIFAR-10 — 60,000 RGB images, 10 classes, 32×32  
**Task:** Train a convolutional neural network from scratch — no pretrained weights

## Why CNNs work for images

A fully connected network on 32×32×3 images would have 3072 input features per neuron — computationally expensive and ignores spatial structure. CNNs solve this with three key ideas:

| Property | What it means |
|---|---|
| Local connectivity | Each filter sees only a small patch of the image |
| Weight sharing | Same filter applied across entire image — fewer parameters |
| Translation invariance | Detecting a cat in the top-left works the same as bottom-right |

## Architecture

```
Input (B, 3, 32, 32)
→ ConvBlock 1: Conv(3→32) + BN + ReLU + Conv(32→32) + BN + ReLU + MaxPool + Dropout  → (B, 32, 16, 16)
→ ConvBlock 2: Conv(32→64) + BN + ReLU + Conv(64→64) + BN + ReLU + MaxPool + Dropout → (B, 64, 8, 8)
→ ConvBlock 3: Conv(64→128) + BN + ReLU + Conv(128→128) + BN + ReLU + MaxPool + Dropout → (B, 128, 4, 4)
→ Flatten → Linear(2048→512) + ReLU + Dropout → Linear(512→10)
```

## Loss Function

**Cross-Entropy Loss** — standard for multi-class classification:

$$\mathcal{L} = -\frac{1}{N}\sum_{i=1}^{N} \log \frac{e^{z_{y_i}}}{\sum_{j} e^{z_j}}$$

where $z_{y_i}$ is the logit for the correct class.

## BatchNorm

Normalizes activations within each mini-batch:

$$\hat{x} = \frac{x - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}, \quad y = \gamma \hat{x} + \beta$$

Benefits: faster convergence, higher learning rates, acts as regularizer.

## CosineAnnealingLR

Learning rate follows a cosine schedule from $lr_{max}$ to $lr_{min}$:

$$lr_t = lr_{min} + \frac{1}{2}(lr_{max} - lr_{min})\left(1 + \cos\frac{t\pi}{T_{max}}\right)$$

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

import yaml
import torch
import numpy as np
import matplotlib.pyplot as plt
import torchvision

from src.cnn.model   import CustomCNN
from src.cnn.trainer import CNNTrainer

with open('../configs/cnn_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print('Config:', config)

In [ ]:
# Model summary and shape verification
model = CustomCNN(
    num_classes = config['model']['num_classes'],
    dropout     = config['model']['dropout'],
).to(device)

print(model)
print(f'\nTotal trainable parameters: {model.count_parameters():,}')

# verify forward pass
x_test = torch.rand(4, 3, 32, 32).to(device)
out    = model(x_test)
print(f'\nInput  : {x_test.shape}')
print(f'Output : {out.shape}  ← should be (4, 10)')

In [ ]:
# Visualize sample CIFAR-10 images
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])
dataset = datasets.CIFAR10(root='../data', train=True, download=True, transform=transform)
loader  = DataLoader(dataset, batch_size=16, shuffle=True)
imgs, labels = next(iter(loader))

classes = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']

# unnormalize for display
mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(3,1,1)
std  = torch.tensor([0.2470, 0.2435, 0.2616]).view(3,1,1)
imgs_display = imgs * std + mean

grid = torchvision.utils.make_grid(imgs_display, nrow=8)
fig, ax = plt.subplots(figsize=(14, 4))
ax.imshow(grid.permute(1,2,0).numpy())
ax.axis('off')
ax.set_title('CIFAR-10 Sample Images — ' + '  '.join([classes[l] for l in labels[:8].tolist()]))
plt.tight_layout()
plt.show()

In [ ]:
# Train the model
# CPU: ~5-8 mins for 15 epochs at batch_size=64
trainer = CNNTrainer(config)
trainer.train()

history = trainer.get_history()
print(f'\nFinal Train Acc : {history["train_accs"][-1]*100:.2f}%')
print(f'Final Val Acc   : {history["val_accs"][-1]*100:.2f}%')

In [ ]:
# Plot training curves
history = trainer.get_history()
epochs  = range(1, len(history['train_losses']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(epochs, history['train_losses'], label='Train', color='#3498DB')
ax1.plot(epochs, history['val_losses'],   label='Val',   color='#E74C3C')
ax1.set_title('Loss'); ax1.set_xlabel('Epoch')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs, [a*100 for a in history['train_accs']], label='Train', color='#3498DB')
ax2.plot(epochs, [a*100 for a in history['val_accs']],   label='Val',   color='#E74C3C')
ax2.set_title('Accuracy (%)'); ax2.set_xlabel('Epoch')
ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle('Custom CNN — CIFAR-10 Training Curves', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate on test set — accuracy + confusion matrix
acc, cm = trainer.evaluate()
print(f'\nTest Accuracy: {acc*100:.2f}%')

# per-class accuracy
classes = trainer.classes
per_class = cm.diagonal() / cm.sum(axis=1)
print('\nPer-class accuracy:')
for cls, acc_c in zip(classes, per_class):
    print(f'  {cls:<12}: {acc_c*100:.1f}%')

## Results

| Metric | Value |
|---|---|
| Test Accuracy | *fill after training* |
| Best Val Accuracy | *fill after training* |
| Parameters | ~1.2M |
| Epochs | 15 |
| Optimizer | Adam + CosineAnnealingLR |

## Key Observations

**BatchNorm** stabilizes training — without it, deeper CNNs suffer from internal covariate shift where distribution of activations changes each layer.  
**Dropout2d** drops entire feature maps during training — stronger regularization than standard dropout for conv layers.  
**MaxPool** reduces spatial dimensions by 2× each block — compresses spatial info while retaining strongest activations.